# None-f Repair Analyzer

In [ ]:
import pandas as pd

# You can read the already calculated repairs file, or read the one generated on 't-box repairs analyzer' folder with the script 't-box_repairs_analyzer.py'
repairs = pd.read_csv("../../datasets/none_of_repairs.csv")

repairs

- The cell below counts different types of basic T-box repairs generated with the relational database:

In [ ]:
# T-box changes coming from the relational db
print(len(repairs[(repairs['C_deleted'] == True)]))
print(len(repairs[(repairs['C_deprecated'] == True)]))
print(len(repairs[(repairs['CQ_added_exception'] == True)]))
print(len(repairs[(repairs['CQ_removed_value'] == True)]))

- check for base statement deletions:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def isRemovedWithObj(subject, property, obj):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    property = property.replace('http://www.wikidata.org/entity/','http://www.wikidata.org/prop/direct/')
    # SPARQL query
    query = f"ASK {{ <{subject}> <{property}> <{obj}> }}"

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

In [ ]:
repairs['S_deleted'] = False

In [ ]:
from tqdm import tqdm

# Wrap the dataframe with tqdm to show progress
for index, row in tqdm(repairs.iterrows(), total=len(repairs), desc="Processing rows"):
    if row['V'].startswith('http'):
        obj = row['V']

        wdtStmtRemoved = isRemovedWithObj(row['subject'], row['property'], obj)

        # Update instanceRemoved column
        repairs.at[index, 'S_deleted'] = wdtStmtRemoved

In [ ]:
len(repairs[(repairs['S_deleted'] == True)])

- test for object replacement:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getSQStatement(subject, property, object):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2019"

    # Prepare the property
    pid = property.replace('http://www.wikidata.org/entity/', '')

    # SPARQL query
    query = f"""
    PREFIX p: <http://www.wikidata.org/prop/>
    PREFIX ps: <http://www.wikidata.org/prop/statement/>
    SELECT ?SQ {{
        <{subject}> p:{pid} ?SQ.
        ?SQ ps:{pid} <{object}>
    }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)

    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"

    # Send HTTP GET request
    headers = {"Accept": "application/sparql-results+xml"}

    response = requests.get(url, headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        root = ET.fromstring(response.text)
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Collect all uri and literal elements
        uris = [uri.text for uri in root.findall('.//ns:uri', namespace)]
        literals = [lit.text for lit in root.findall('.//ns:literal', namespace)]

        results = uris + literals  # Combine both types of results

        return results if results else None
    else:
        print("Error:", response.text)
        return None

def getObjectsOfStatementPS(subject, property):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # Prepare the property
    pid = property.replace('http://www.wikidata.org/entity/', '')

    # SPARQL query
    query = f"""
    SELECT ?o {{
        <{subject}> ps:{pid} ?o
    }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)

    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"

    # Send HTTP GET request
    headers = {"Accept": "application/sparql-results+xml"}

    response = requests.get(url, headers=headers)

    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        root = ET.fromstring(response.text)
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Collect all uri and literal elements
        uris = [uri.text for uri in root.findall('.//ns:uri', namespace)]
        literals = [lit.text for lit in root.findall('.//ns:literal', namespace)]

        results = uris + literals  # Combine both types of results

        return results if results else None
    else:
        print("Error:", response.text)
        return None

In [ ]:
repairs['Sr_replacement'] = None

In [ ]:
from tqdm import tqdm
import os

for index, row in tqdm(repairs.iterrows(), total=len(repairs)):
 
    if row['Sr_replacement'] is None:
        if row['S_deleted'] is True:
            repairs.at[index, 'Sr_replacement'] = False
        else:
            CQ_list = getSQStatement(row['subject'], row['property'], row['V'])
            obj_list = getObjectsOfStatementPS(CQ_list[0], row['property'])
            if obj_list is None or row['V'] in obj_list:
                repairs.at[index, 'Sr_replacement'] = False
            else:
                repairs.at[index, 'Sr_replacement'] = True

- generate Venn diagram:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Create the new DataFrame with the required columns
df2 = pd.DataFrame()
df2['A-box changes'] = repairs['S_deleted'] | repairs['Sr_replacement']
df2['T-box changes'] = (
    repairs['C_deleted'] | 
    repairs['C_deprecated'] | 
    repairs['CQ_added_exception'] | 
    repairs['CQ_removed_value']
)

# Calculate the sizes of the sets
a_box_changes = df2['A-box changes'].sum()
t_box_changes = df2['T-box changes'].sum()
intersection = (df2['A-box changes'] & df2['T-box changes']).sum()

# Plot the Venn diagram
venn2(subsets=(a_box_changes, t_box_changes, intersection), 
      set_labels=('A-box changes', 'T-box changes'))


# Add title
plt.title("None-of Constraint share of repairs")

plt.show()

In [ ]:
repairs.to_csv('none_of_repairs_final.csv', index=False)